# 100 — Export virtual T1 survey to SEG-Y (SAFE, v2)

This notebook exports notebook 99's virtual T1 MiniSEED shot gathers in strict
source-position order.

## Products

### Sparse authoritative survey

Contains only observed virtual traces:

```text
source position ascending
    receiver position ascending
        receiver family
```

### Regularized convenience survey

Contains the union of all receiver-family/receiver-position keys at every shot.
Missing combinations are zero-filled and marked as dead traces in the SEG-Y
trace-identification header.

### Optional per-shot SEG-Y

Useful for inspection and software that expects one shot gather per file.

The CSV catalogs produced by notebooks 99 and 100 remain authoritative because
SEG-Y headers cannot represent the full provenance graph.

## 1. Configuration

In [1]:
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

from obspy import read, Stream, Trace
from obspy.io.segy.segy import SEGYTraceHeader, SEGYBinaryFileHeader
from obspy.core import AttribDict

PROJECT_ROOT = Path('/Volumes/tachyon/LBSSP_DATA')
INPUT_ROOT = PROJECT_ROOT / '99_virtual_T1_shot_gathers'
OUT_ROOT = PROJECT_ROOT / '100_virtual_T1_SEGY'
PER_SHOT_ROOT = OUT_ROOT / 'per_shot_segy'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
PER_SHOT_ROOT.mkdir(parents=True, exist_ok=True)

SHOT_MANIFEST_PATH = INPUT_ROOT / '99_virtual_T1_shot_manifest.csv'
TRACE_MANIFEST_PATH = INPUT_ROOT / '99_virtual_T1_trace_manifest.csv'

COMPONENT = 'Z'
DATA_ENCODING = 5  # IEEE 32-bit float
BYTEORDER = '>'
WRITE_PER_SHOT_SEGY = True
WRITE_SPARSE_ALL_SHOTS = True
WRITE_REGULARIZED_ALL_SHOTS = True
REQUIRE_GEODE_TRACES = True

# Coordinates are stored as integer millimetres with scalar -1000.
COORDINATE_SCALAR = -1000
COORDINATE_MULTIPLIER = 1000

pd.set_option('display.max_columns', 300)
pd.set_option('display.width', 280)

print('Input:', INPUT_ROOT)
print('Output:', OUT_ROOT)

Input: /Volumes/tachyon/LBSSP_DATA/99_virtual_T1_shot_gathers
Output: /Volumes/tachyon/LBSSP_DATA/100_virtual_T1_SEGY


## 2. Load ordered shot and trace catalogs

In [2]:
for path in [SHOT_MANIFEST_PATH, TRACE_MANIFEST_PATH]:
    if not path.exists():
        raise FileNotFoundError(f'Missing notebook-99 output: {path}')

shots = pd.read_csv(SHOT_MANIFEST_PATH, low_memory=False)
traces = pd.read_csv(TRACE_MANIFEST_PATH, low_memory=False)

shots = shots.loc[
    shots.status.eq('written')
    & shots.component.astype(str).str.upper().eq(COMPONENT)
].copy()

shots = shots.sort_values(
    ['source_x_m', 'virtual_shot_number'],
    kind='stable',
).reset_index(drop=True)

traces['source_x_m'] = pd.to_numeric(
    traces.source_x_m, errors='coerce'
)
traces['receiver_x_m'] = pd.to_numeric(
    traces.receiver_x_m, errors='coerce'
)

print('Shots to export:', len(shots))
print('Observed traces:', len(traces))
receiver_family_counts = traces.receiver_family.value_counts(dropna=False)
display(receiver_family_counts.rename_axis('receiver_family').reset_index(name='n_traces'))
if REQUIRE_GEODE_TRACES and not traces.receiver_family.astype(str).eq('geode').any():
    raise RuntimeError(
        'No Geode traces are present in the notebook-99 trace manifest. '
        'Rerun updated notebooks 98 and 99 before exporting SEG-Y.'
    )

Shots to export: 159
Observed traces: 13106


,receiver_family,n_traces
0,geode,7206
1,nodal,5900


## 3. SEG-Y header helpers

In [3]:
def coordinate_integer(value_m):
    return int(round(float(value_m) * COORDINATE_MULTIPLIER))


def attach_segy_header(
    trace,
    *,
    global_trace_number,
    shot_number,
    trace_number_within_shot,
    source_x_m,
    receiver_x_m,
    is_live,
):
    trace.stats.segy = AttribDict()
    header = SEGYTraceHeader()

    header.trace_sequence_number_within_line = int(
        global_trace_number
    )
    header.trace_sequence_number_within_segy_file = int(
        global_trace_number
    )
    header.original_field_record_number = int(shot_number)
    header.trace_number_within_the_original_field_record = int(
        trace_number_within_shot
    )
    header.energy_source_point_number = int(shot_number)

    header.scalar_to_be_applied_to_all_coordinates = (
        COORDINATE_SCALAR
    )
    header.coordinate_units = 1
    header.source_coordinate_x = coordinate_integer(source_x_m)
    header.group_coordinate_x = coordinate_integer(receiver_x_m)
    header.distance_from_center_of_the_source_point_to_the_center_of_the_receiver_group = (
        coordinate_integer(receiver_x_m - source_x_m)
    )

    # 1 = seismic data, 2 = dead trace.
    header.trace_identification_code = 1 if is_live else 2

    trace.stats.segy.trace_header = header
    return trace


def write_segy(
    stream,
    path,
    *,
    traces_per_ensemble=0,
):
    """
    Write SEG-Y while explicitly setting the binary-header number of data
    traces per ensemble.

    ObsPy otherwise uses the total number of traces in the Stream. For an
    all-shot file that can exceed the signed 16-bit SEG-Y field, even though
    the actual number of traces in each shot ensemble is much smaller.
    """
    if not len(stream):
        return

    traces_per_ensemble = int(traces_per_ensemble)
    if not 0 <= traces_per_ensemble <= 32767:
        raise ValueError(
            'traces_per_ensemble must fit the signed 16-bit SEG-Y '
            f'header field; got {traces_per_ensemble}.'
        )

    stream.stats = AttribDict()
    stream.stats.binary_file_header = SEGYBinaryFileHeader()
    stream.stats.binary_file_header.number_of_data_traces_per_ensemble = (
        traces_per_ensemble
    )
    stream.stats.binary_file_header.number_of_auxiliary_traces_per_ensemble = 0

    stream.write(
        str(path),
        format='SEGY',
        data_encoding=DATA_ENCODING,
        byteorder=BYTEORDER,
    )

## 4. Read shot gathers and construct sparse SEG-Y stream

In [4]:
sparse_stream = Stream()
sparse_catalog_rows = []
global_trace_number = 0
per_shot_streams = {}

for shot in shots.itertuples(index=False):
    path = Path(str(shot.mseed_path))
    if not path.exists():
        print('WARNING: missing MiniSEED:', path)
        continue

    stream = read(str(path))
    shot_trace_catalog = traces.loc[
        traces.virtual_shot_number.eq(shot.virtual_shot_number)
    ].copy()

    lookup = {
        (
            str(row.station),
            str(row.channel),
        ): row
        for row in shot_trace_catalog.itertuples(index=False)
    }

    prepared = []

    for trace in stream:
        key = (str(trace.stats.station), str(trace.stats.channel))
        row = lookup.get(key)
        if row is None:
            continue
        prepared.append((row, trace.copy()))

    prepared.sort(
        key=lambda item: (
            float(item[0].receiver_x_m),
            str(item[0].receiver_family),
            str(item[0].station),
        )
    )

    shot_stream = Stream()

    for trace_number, (row, trace) in enumerate(prepared, start=1):
        global_trace_number += 1
        trace.data = np.asarray(trace.data, dtype=np.float32)
        attach_segy_header(
            trace,
            global_trace_number=global_trace_number,
            shot_number=int(shot.virtual_shot_number),
            trace_number_within_shot=trace_number,
            source_x_m=float(shot.source_x_m),
            receiver_x_m=float(row.receiver_x_m),
            is_live=True,
        )
        shot_stream += trace
        sparse_stream += trace.copy()

        sparse_catalog_rows.append({
            'global_trace_number': global_trace_number,
            'virtual_shot_number': int(shot.virtual_shot_number),
            'virtual_shot_id': shot.virtual_shot_id,
            'source_x_m': float(shot.source_x_m),
            'trace_number_within_shot': trace_number,
            'receiver_family': row.receiver_family,
            'receiver_x_m': float(row.receiver_x_m),
            'station': row.station,
            'channel': row.channel,
            'trace_identification_code': 1,
            'is_live': True,
        })

    per_shot_streams[int(shot.virtual_shot_number)] = shot_stream

print('Sparse SEG-Y traces:', len(sparse_stream))

Sparse SEG-Y traces: 13106


## 5. Write sparse all-shot and per-shot SEG-Y

In [5]:
SPARSE_PATH = OUT_ROOT / f'T1_virtual_sparse_{COMPONENT}.segy'

if WRITE_SPARSE_ALL_SHOTS:
    write_segy(
        sparse_stream,
        SPARSE_PATH,
        traces_per_ensemble=0,
    )
    print('Wrote:', SPARSE_PATH)

if WRITE_PER_SHOT_SEGY:
    for shot in shots.itertuples(index=False):
        shot_stream = per_shot_streams.get(
            int(shot.virtual_shot_number),
            Stream(),
        )
        if not len(shot_stream):
            continue
        filename = (
            f'T1_VIRTUAL_SHOT_{int(shot.virtual_shot_number):04d}'
            f'_x{float(shot.source_x_m):07.1f}m_{COMPONENT}.segy'
        )
        write_segy(
            shot_stream,
            PER_SHOT_ROOT / filename,
            traces_per_ensemble=len(shot_stream),
        )

    print('Per-shot SEG-Y directory:', PER_SHOT_ROOT)

Wrote: /Volumes/tachyon/LBSSP_DATA/100_virtual_T1_SEGY/T1_virtual_sparse_Z.segy
Per-shot SEG-Y directory: /Volumes/tachyon/LBSSP_DATA/100_virtual_T1_SEGY/per_shot_segy


## 6. Build regularized receiver grid

In [6]:
master_receivers = (
    traces[
        ['receiver_family', 'receiver_x_m', 'channel']
    ]
    .drop_duplicates()
    .sort_values(
        ['receiver_x_m', 'receiver_family', 'channel'],
        kind='stable',
    )
    .reset_index(drop=True)
)

master_receivers.insert(
    0,
    'master_receiver_number',
    np.arange(1, len(master_receivers) + 1, dtype=int),
)

print('Master receiver keys:', len(master_receivers))
display(master_receivers.head(30))

Master receiver keys: 242


,master_receiver_number,receiver_family,receiver_x_m,channel
0,1,geode,0.00,GHZ
1,2,geode,5.00,GHZ
2,3,geode,10.00,GHZ
3,4,geode,15.00,GHZ
4,5,geode,20.00,GHZ
5,6,geode,25.00,GHZ
6,7,nodal,28.00,NHZ
7,8,geode,30.00,GHZ
8,9,geode,35.00,GHZ
9,10,nodal,36.00,NHZ


## 7. Write regularized all-shot SEG-Y with dead traces

In [7]:
regularized_stream = Stream()
regularized_catalog_rows = []
global_trace_number = 0

if WRITE_REGULARIZED_ALL_SHOTS:
    for shot in shots.itertuples(index=False):
        shot_stream = per_shot_streams.get(
            int(shot.virtual_shot_number),
            Stream(),
        )

        observed_lookup = {}
        shot_trace_catalog = traces.loc[
            traces.virtual_shot_number.eq(
                shot.virtual_shot_number
            )
        ]

        for row in shot_trace_catalog.itertuples(index=False):
            candidates = [
                trace
                for trace in shot_stream
                if (
                    str(trace.stats.station) == str(row.station)
                    and str(trace.stats.channel) == str(row.channel)
                )
            ]
            if candidates:
                key = (
                    str(row.receiver_family),
                    round(float(row.receiver_x_m), 6),
                    str(row.channel),
                )
                observed_lookup[key] = candidates[0].copy()

        if len(shot_stream):
            template = shot_stream[0]
        else:
            raise RuntimeError(
                f'No template trace available for shot {shot.virtual_shot_number}'
            )

        for trace_number, receiver in enumerate(
            master_receivers.itertuples(index=False),
            start=1,
        ):
            key = (
                str(receiver.receiver_family),
                round(float(receiver.receiver_x_m), 6),
                str(receiver.channel),
            )

            is_live = key in observed_lookup
            if is_live:
                trace = observed_lookup[key].copy()
            else:
                trace = Trace(
                    data=np.zeros(
                        template.stats.npts,
                        dtype=np.float32,
                    )
                )
                trace.stats.sampling_rate = (
                    template.stats.sampling_rate
                )
                trace.stats.starttime = template.stats.starttime
                trace.stats.network = 'VT'
                trace.stats.station = (
                    f'D{int(receiver.master_receiver_number):04d}'[:5]
                )
                trace.stats.location = 'DD'
                trace.stats.channel = str(receiver.channel)

            global_trace_number += 1
            trace.data = np.asarray(trace.data, dtype=np.float32)

            attach_segy_header(
                trace,
                global_trace_number=global_trace_number,
                shot_number=int(shot.virtual_shot_number),
                trace_number_within_shot=trace_number,
                source_x_m=float(shot.source_x_m),
                receiver_x_m=float(receiver.receiver_x_m),
                is_live=is_live,
            )

            regularized_stream += trace

            regularized_catalog_rows.append({
                'global_trace_number': global_trace_number,
                'virtual_shot_number': int(
                    shot.virtual_shot_number
                ),
                'virtual_shot_id': shot.virtual_shot_id,
                'source_x_m': float(shot.source_x_m),
                'trace_number_within_shot': trace_number,
                'master_receiver_number': int(
                    receiver.master_receiver_number
                ),
                'receiver_family': receiver.receiver_family,
                'receiver_x_m': float(receiver.receiver_x_m),
                'channel': receiver.channel,
                'trace_identification_code': 1 if is_live else 2,
                'is_live': is_live,
            })

    REGULARIZED_PATH = (
        OUT_ROOT / f'T1_virtual_regularized_{COMPONENT}.segy'
    )
    write_segy(
        regularized_stream,
        REGULARIZED_PATH,
        traces_per_ensemble=len(master_receivers),
    )
    print('Wrote:', REGULARIZED_PATH)
    print('Regularized traces:', len(regularized_stream))

Wrote: /Volumes/tachyon/LBSSP_DATA/100_virtual_T1_SEGY/T1_virtual_regularized_Z.segy
Regularized traces: 38478


## 8. Export SEG-Y catalogs and summary

In [8]:
sparse_catalog = pd.DataFrame(sparse_catalog_rows)
regularized_catalog = pd.DataFrame(regularized_catalog_rows)

OUTPUTS = {
    'sparse_catalog': OUT_ROOT / '100_virtual_T1_sparse_trace_catalog.csv',
    'master_receivers': OUT_ROOT / '100_virtual_T1_master_receiver_grid.csv',
    'regularized_catalog': OUT_ROOT / '100_virtual_T1_regularized_trace_catalog.csv',
    'summary': OUT_ROOT / '100_virtual_T1_SEGY_summary.csv',
}

sparse_catalog.to_csv(OUTPUTS['sparse_catalog'], index=False)
master_receivers.to_csv(OUTPUTS['master_receivers'], index=False)
regularized_catalog.to_csv(
    OUTPUTS['regularized_catalog'], index=False
)

summary = pd.DataFrame([
    ('shots_exported', len(shots)),
    ('sparse_live_traces', len(sparse_catalog)),
    ('sparse_nodal_traces', int(sparse_catalog.receiver_family.eq('nodal').sum()) if len(sparse_catalog) else 0),
    ('sparse_geode_traces', int(sparse_catalog.receiver_family.eq('geode').sum()) if len(sparse_catalog) else 0),
    ('master_receiver_keys', len(master_receivers)),
    ('binary_header_traces_per_ensemble', len(master_receivers)),
    ('regularized_total_traces', len(regularized_catalog)),
    ('regularized_live_traces', int(
        regularized_catalog.is_live.sum()
    ) if len(regularized_catalog) else 0),
    ('regularized_dead_traces', int(
        (~regularized_catalog.is_live).sum()
    ) if len(regularized_catalog) else 0),
], columns=['metric', 'value'])
summary.to_csv(OUTPUTS['summary'], index=False)
display(summary)

print('\nWritten:')
for name, path in OUTPUTS.items():
    print(f'  {name:20s} {path}')

,metric,value
0,shots_exported,159
1,sparse_live_traces,13106
2,sparse_nodal_traces,5900
3,sparse_geode_traces,7206
4,master_receiver_keys,242
5,binary_header_traces_per_ensemble,242
6,regularized_total_traces,38478
7,regularized_live_traces,16615
8,regularized_dead_traces,21863



Written:
  sparse_catalog       /Volumes/tachyon/LBSSP_DATA/100_virtual_T1_SEGY/100_virtual_T1_sparse_trace_catalog.csv
  master_receivers     /Volumes/tachyon/LBSSP_DATA/100_virtual_T1_SEGY/100_virtual_T1_master_receiver_grid.csv
  regularized_catalog  /Volumes/tachyon/LBSSP_DATA/100_virtual_T1_SEGY/100_virtual_T1_regularized_trace_catalog.csv
  summary              /Volumes/tachyon/LBSSP_DATA/100_virtual_T1_SEGY/100_virtual_T1_SEGY_summary.csv


## 9. Interpretation

Use `T1_virtual_sparse_Z.segy` as the authoritative observed survey.

Use `T1_virtual_regularized_Z.segy` only when downstream software requires a
rectangular source-receiver matrix. A zero-valued trace with SEG-Y
`trace_identification_code = 2` is missing/dead data, not an observed zero
amplitude.

The trace catalogs preserve receiver family and live/dead status explicitly.